# Middle East conflict experiments

This notebook demonstrates how to run experiments with `pareto_conf_general.py` and Middle East conflict example.

It covers:
- loading the model from CSV,
- enumerating coalitions, oppositions, and bi-conflicts,
- comparing frontiers under **Q** and **Q***,
- switching semantic assumptions,
- converting results to pandas DataFrames,
- exporting results to CSV.


### Files expected in the DATA_DIR folder (or update the paths below)
The files expected content is described in pareto_conf_general script
- `attitudes.csv`
- `agent_classes.csv`
- `issue_classes.csv`
- `agent_names.csv`
- `issue_names.csv`


In [25]:
from pathlib import Path
import sys

# Adjust these if your notebook is stored in a different location.
PROJECT_DIR = Path('.').resolve()
DATA_DIR = PROJECT_DIR / 'MiddleEast'

# Make sure the module can be imported.
if str(PROJECT_DIR) not in sys.path:
    sys.path.append(str(PROJECT_DIR))

print('PROJECT_DIR =', PROJECT_DIR)
print('DATA_DIR    =', DATA_DIR)


PROJECT_DIR = C:\Users\rafal.deja\Documents\Rafal\Praca naukowa\Project\ConflictAnalysis
DATA_DIR    = C:\Users\rafal.deja\Documents\Rafal\Praca naukowa\Project\ConflictAnalysis\MiddleEast


In [26]:
import pandas as pd
import pareto_conf_general as pconf

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)


In [27]:
model = pconf.ConflictModel.from_csv(
    attitudes_csv=DATA_DIR / 'attitudes.csv',
    agent_classes_csv=DATA_DIR / 'agent_classes.csv',
    issue_classes_csv=DATA_DIR / 'issue_classes.csv',
    agent_names_csv=DATA_DIR / 'agent_names.csv',
    issue_names_csv=DATA_DIR / 'issue_names.csv',
    agreement_mode='equal',
    opposition_mode='different',
)

print('Agents       :', model.agents)
print('Issues       :', model.issues)
print('Agent classes:', model.agent_classes)
print('Issue classes:', model.issue_classes)


Agents       : ['x1', 'x2', 'x3', 'x4', 'x5', 'x6']
Issues       : ['i1', 'i2', 'i3', 'i4', 'i5']
Agent classes: [frozenset({'x1', 'x2'}), frozenset({'x3', 'x6'}), frozenset({'x5', 'x4'})]
Issue classes: [frozenset({'i3', 'i1'}), frozenset({'i4', 'i2'}), frozenset({'i5'})]


### Helper functions for DataFrame views


In [28]:
def structures_to_df(structures, model):
    rows = []
    for s in structures:
        rows.append({
            'A_ids': sorted(s.A),
            'B_ids': sorted(s.B),
            'A_names': [model.agent_names.get(a, a) for a in sorted(s.A)],
            'B_names': [model.issue_names.get(i, i) for i in sorted(s.B)],
            'vU': s.vU,
            'vI': s.vI,
            'rU': s.rU,
            'rI': s.rI,
        })
    return pd.DataFrame(rows)

def biconflicts_to_df(biconflicts, model):
    rows = []
    for bc in biconflicts:
        c = bc.coalition
        o = bc.opposition
        rows.append({
            'coalition_A_ids': sorted(c.A),
            'coalition_B_ids': sorted(c.B),
            'coalition_A_names': [model.agent_names.get(a, a) for a in sorted(c.A)],
            'coalition_B_names': [model.issue_names.get(i, i) for i in sorted(c.B)],
            'coalition_vU': c.vU,
            'coalition_vI': c.vI,
            'coalition_rU': c.rU,
            'coalition_rI': c.rI,
            'opposition_A_ids': sorted(o.A),
            'opposition_B_ids': sorted(o.B),
            'opposition_A_names': [model.agent_names.get(a, a) for a in sorted(o.A)],
            'opposition_B_names': [model.issue_names.get(i, i) for i in sorted(o.B)],
            'opposition_vU': o.vU,
            'opposition_vI': o.vI,
            'opposition_rU': o.rU,
            'opposition_rI': o.rI,
        })
    return pd.DataFrame(rows)


### Experiment 1 — enumerate all coalitions


In [29]:
coalitions = model.enumerate_all_coalitions()
front_q = pconf.frontier_Q(coalitions)
front_qstar = pconf.frontier_Q_star(coalitions)

print('All coalitions :', len(coalitions))
print('Frontier Q     :', len(front_q))
print('Frontier Q*    :', len(front_qstar))


All coalitions : 275
Frontier Q     : 5
Frontier Q*    : 3


In [30]:
df_coalitions = structures_to_df(coalitions, model)
df_front_q = structures_to_df(front_q, model)
df_front_qstar = structures_to_df(front_qstar, model)

display(df_coalitions.head(10))
display(df_front_q.sort_values(['rU', 'rI'], ascending=False))
display(df_front_qstar.sort_values(['rU', 'rI'], ascending=False))


,A_ids,B_ids,A_names,B_names,vU,vI,rU,rI
0,[x1],[i1],[Israel],[Autonomous Palestinian state on the West Bank and Gaza],"(1, 0, 0)","(1, 0, 0)",49,36
1,[x1],[i2],[Israel],[Israeli military outposts along the Jordan River],"(1, 0, 0)","(0, 1, 0)",49,6
2,[x1],[i3],[Israel],[Israel retains East Jerusalem],"(1, 0, 0)","(1, 0, 0)",49,36
3,[x1],[i4],[Israel],[Israeli military outposts on the Golan Heights],"(1, 0, 0)","(0, 1, 0)",49,6
4,[x1],[i5],[Israel],[Arab countries grant citizenship to Palestinians who remain within their borders],"(1, 0, 0)","(0, 0, 1)",49,1
5,[x1],"[i1, i2]",[Israel],"[Autonomous Palestinian state on the West Bank and Gaza, Israeli military outposts along the Jordan River]","(1, 0, 0)","(1, 1, 0)",49,42
6,[x1],"[i1, i3]",[Israel],"[Autonomous Palestinian state on the West Bank and Gaza, Israel retains East Jerusalem]","(1, 0, 0)","(2, 0, 0)",49,72
7,[x1],"[i1, i4]",[Israel],"[Autonomous Palestinian state on the West Bank and Gaza, Israeli military outposts on the Golan Heights]","(1, 0, 0)","(1, 1, 0)",49,42
8,[x1],"[i1, i5]",[Israel],"[Autonomous Palestinian state on the West Bank and Gaza, Arab countries grant citizenship to Palestinians who remain...","(1, 0, 0)","(1, 0, 1)",49,37
9,[x1],"[i2, i3]",[Israel],"[Israeli military outposts along the Jordan River, Israel retains East Jerusalem]","(1, 0, 0)","(1, 1, 0)",49,42


,A_ids,B_ids,A_names,B_names,vU,vI,rU,rI
0,"[x2, x3, x4, x5, x6]",[i3],"[Egypt, Palestine, Jordan, Syria, Saudi Arabia]",[Israel retains East Jerusalem],"(1, 2, 2)","(1, 0, 0)",65,36
1,"[x2, x3, x5]","[i1, i3, i4]","[Egypt, Palestine, Syria]","[Autonomous Palestinian state on the West Bank and Gaza, Israel retains East Jerusalem, Israeli military outposts on...","(1, 1, 1)","(2, 1, 0)",57,78
2,"[x2, x5]","[i1, i3, i4, i5]","[Egypt, Syria]","[Autonomous Palestinian state on the West Bank and Gaza, Israel retains East Jerusalem, Israeli military outposts on...","(1, 0, 1)","(2, 1, 1)",50,79
3,[x1],"[i1, i2, i3, i4, i5]",[Israel],"[Autonomous Palestinian state on the West Bank and Gaza, Israeli military outposts along the Jordan River, Israel re...","(1, 0, 0)","(2, 2, 1)",49,85
4,[x2],"[i1, i2, i3, i4, i5]",[Egypt],"[Autonomous Palestinian state on the West Bank and Gaza, Israeli military outposts along the Jordan River, Israel re...","(1, 0, 0)","(2, 2, 1)",49,85


,A_ids,B_ids,A_names,B_names,vU,vI,rU,rI
2,"[x2, x3, x5]","[i1, i3, i4]","[Egypt, Palestine, Syria]","[Autonomous Palestinian state on the West Bank and Gaza, Israel retains East Jerusalem, Israeli military outposts on...","(1, 1, 1)","(2, 1, 0)",57,78
0,[x1],"[i1, i2, i3, i4, i5]",[Israel],"[Autonomous Palestinian state on the West Bank and Gaza, Israeli military outposts along the Jordan River, Israel re...","(1, 0, 0)","(2, 2, 1)",49,85
1,[x2],"[i1, i2, i3, i4, i5]",[Egypt],"[Autonomous Palestinian state on the West Bank and Gaza, Israeli military outposts along the Jordan River, Israel re...","(1, 0, 0)","(2, 2, 1)",49,85


### Experiment 2 — oppositions to a chosen group

Change `G` below to explore different reference groups.


In [31]:
G = frozenset({'x1'})  # example: Israel

oppositions = model.enumerate_oppositions_for_group(G)
opp_front_q = pconf.frontier_Q(oppositions)
opp_front_qstar = pconf.frontier_Q_star(oppositions)

print('Reference group         :', model.fmt_agents(G))
print('Oppositions             :', len(oppositions))
print('Opposition frontier Q   :', len(opp_front_q))
print('Opposition frontier Q*  :', len(opp_front_qstar))


Reference group         : {Israel}
Oppositions             : 31
Opposition frontier Q   : 2
Opposition frontier Q*  : 2


In [32]:
df_oppositions = structures_to_df(oppositions, model)
df_opp_front_q = structures_to_df(opp_front_q, model)
df_opp_front_qstar = structures_to_df(opp_front_qstar, model)

display(df_oppositions.sort_values(['rU', 'rI'], ascending=False))
display(df_opp_front_q.sort_values(['rU', 'rI'], ascending=False))
display(df_opp_front_qstar.sort_values(['rU', 'rI'], ascending=False))


,A_ids,B_ids,A_names,B_names,vU,vI,rU,rI
30,"[x2, x3, x4, x5, x6]","[i1, i3, i4]","[Egypt, Palestine, Jordan, Syria, Saudi Arabia]","[Autonomous Palestinian state on the West Bank and Gaza, Israel retains East Jerusalem, Israeli military outposts on...","(1, 2, 2)","(2, 1, 0)",65,78
26,"[x2, x3, x4, x6]","[i1, i3, i4]","[Egypt, Palestine, Jordan, Saudi Arabia]","[Autonomous Palestinian state on the West Bank and Gaza, Israel retains East Jerusalem, Israeli military outposts on...","(1, 2, 1)","(2, 1, 0)",64,78
27,"[x2, x3, x5, x6]","[i1, i3, i4]","[Egypt, Palestine, Syria, Saudi Arabia]","[Autonomous Palestinian state on the West Bank and Gaza, Israel retains East Jerusalem, Israeli military outposts on...","(1, 2, 1)","(2, 1, 0)",64,78
17,"[x2, x3, x6]","[i1, i3, i4]","[Egypt, Palestine, Saudi Arabia]","[Autonomous Palestinian state on the West Bank and Gaza, Israel retains East Jerusalem, Israeli military outposts on...","(1, 2, 0)","(2, 1, 0)",63,78
25,"[x2, x3, x4, x5]","[i1, i2, i3, i4, i5]","[Egypt, Palestine, Jordan, Syria]","[Autonomous Palestinian state on the West Bank and Gaza, Israeli military outposts along the Jordan River, Israel re...","(1, 1, 2)","(2, 2, 1)",58,85
28,"[x2, x4, x5, x6]","[i1, i3, i4]","[Egypt, Jordan, Syria, Saudi Arabia]","[Autonomous Palestinian state on the West Bank and Gaza, Israel retains East Jerusalem, Israeli military outposts on...","(1, 1, 2)","(2, 1, 0)",58,78
15,"[x2, x3, x4]","[i1, i2, i3, i4, i5]","[Egypt, Palestine, Jordan]","[Autonomous Palestinian state on the West Bank and Gaza, Israeli military outposts along the Jordan River, Israel re...","(1, 1, 1)","(2, 2, 1)",57,85
16,"[x2, x3, x5]","[i1, i2, i3, i4, i5]","[Egypt, Palestine, Syria]","[Autonomous Palestinian state on the West Bank and Gaza, Israeli military outposts along the Jordan River, Israel re...","(1, 1, 1)","(2, 2, 1)",57,85
19,"[x2, x4, x6]","[i1, i3, i4]","[Egypt, Jordan, Saudi Arabia]","[Autonomous Palestinian state on the West Bank and Gaza, Israel retains East Jerusalem, Israeli military outposts on...","(1, 1, 1)","(2, 1, 0)",57,78
20,"[x2, x5, x6]","[i1, i3, i4]","[Egypt, Syria, Saudi Arabia]","[Autonomous Palestinian state on the West Bank and Gaza, Israel retains East Jerusalem, Israeli military outposts on...","(1, 1, 1)","(2, 1, 0)",57,78


,A_ids,B_ids,A_names,B_names,vU,vI,rU,rI
0,"[x2, x3, x4, x5, x6]","[i1, i3, i4]","[Egypt, Palestine, Jordan, Syria, Saudi Arabia]","[Autonomous Palestinian state on the West Bank and Gaza, Israel retains East Jerusalem, Israeli military outposts on...","(1, 2, 2)","(2, 1, 0)",65,78
1,"[x2, x3, x4, x5]","[i1, i2, i3, i4, i5]","[Egypt, Palestine, Jordan, Syria]","[Autonomous Palestinian state on the West Bank and Gaza, Israeli military outposts along the Jordan River, Israel re...","(1, 1, 2)","(2, 2, 1)",58,85


,A_ids,B_ids,A_names,B_names,vU,vI,rU,rI
1,"[x2, x3, x4, x5, x6]","[i1, i3, i4]","[Egypt, Palestine, Jordan, Syria, Saudi Arabia]","[Autonomous Palestinian state on the West Bank and Gaza, Israel retains East Jerusalem, Israeli military outposts on...","(1, 2, 2)","(2, 1, 0)",65,78
0,"[x2, x3, x4, x5]","[i1, i2, i3, i4, i5]","[Egypt, Palestine, Jordan, Syria]","[Autonomous Palestinian state on the West Bank and Gaza, Israeli military outposts along the Jordan River, Israel re...","(1, 1, 2)","(2, 2, 1)",58,85


### Experiment 3 — bi-conflicts for the given coalition


In [33]:
G = frozenset({"x2", "x3",  "x5"})
biconflicts = model.enumerate_biconflicts_for_group(G)
bi_front_q = pconf.biconflict_frontier_Q(biconflicts)
bi_front_qstar = pconf.biconflict_frontier_Q_star(biconflicts)

print('Bi-conflicts            :', len(biconflicts))
print('Bi-conflict frontier Q  :', len(bi_front_q))
print('Bi-conflict frontier Q* :', len(bi_front_qstar))


Bi-conflicts            : 7
Bi-conflict frontier Q  : 3
Bi-conflict frontier Q* : 1


In [34]:
df_biconflicts = biconflicts_to_df(biconflicts, model)
df_bi_front_q = biconflicts_to_df(bi_front_q, model)
df_bi_front_qstar = biconflicts_to_df(bi_front_qstar, model)

display(df_biconflicts.head(10))
display(df_bi_front_q.head(10))
display(df_bi_front_qstar.head(10))


,coalition_A_ids,coalition_B_ids,coalition_A_names,coalition_B_names,coalition_vU,coalition_vI,coalition_rU,coalition_rI,opposition_A_ids,opposition_B_ids,opposition_A_names,opposition_B_names,opposition_vU,opposition_vI,opposition_rU,opposition_rI
0,"[x2, x3, x5]","[i1, i3, i4]","[Egypt, Palestine, Syria]","[Autonomous Palestinian state on the West Bank and Gaza, Israel retains East Jerusalem, Israeli military outposts on...","(1, 1, 1)","(2, 1, 0)",57,78,[x1],"[i1, i2, i3, i4, i5]",[Israel],"[Autonomous Palestinian state on the West Bank and Gaza, Israeli military outposts along the Jordan River, Israel re...","(1, 0, 0)","(2, 2, 1)",49,85
1,"[x2, x3, x5]","[i1, i3, i4]","[Egypt, Palestine, Syria]","[Autonomous Palestinian state on the West Bank and Gaza, Israel retains East Jerusalem, Israeli military outposts on...","(1, 1, 1)","(2, 1, 0)",57,78,[x4],"[i1, i4]",[Jordan],"[Autonomous Palestinian state on the West Bank and Gaza, Israeli military outposts on the Golan Heights]","(0, 0, 1)","(1, 1, 0)",1,42
2,"[x2, x3, x5]","[i1, i3, i4]","[Egypt, Palestine, Syria]","[Autonomous Palestinian state on the West Bank and Gaza, Israel retains East Jerusalem, Israeli military outposts on...","(1, 1, 1)","(2, 1, 0)",57,78,[x6],"[i1, i2, i4, i5]",[Saudi Arabia],"[Autonomous Palestinian state on the West Bank and Gaza, Israeli military outposts along the Jordan River, Israeli m...","(0, 1, 0)","(1, 2, 1)",7,49
3,"[x2, x3, x5]","[i1, i3, i4]","[Egypt, Palestine, Syria]","[Autonomous Palestinian state on the West Bank and Gaza, Israel retains East Jerusalem, Israeli military outposts on...","(1, 1, 1)","(2, 1, 0)",57,78,"[x1, x4]","[i1, i4]","[Israel, Jordan]","[Autonomous Palestinian state on the West Bank and Gaza, Israeli military outposts on the Golan Heights]","(1, 0, 1)","(1, 1, 0)",50,42
4,"[x2, x3, x5]","[i1, i3, i4]","[Egypt, Palestine, Syria]","[Autonomous Palestinian state on the West Bank and Gaza, Israel retains East Jerusalem, Israeli military outposts on...","(1, 1, 1)","(2, 1, 0)",57,78,"[x1, x6]","[i1, i2, i4, i5]","[Israel, Saudi Arabia]","[Autonomous Palestinian state on the West Bank and Gaza, Israeli military outposts along the Jordan River, Israeli m...","(1, 1, 0)","(1, 2, 1)",56,49
5,"[x2, x3, x5]","[i1, i3, i4]","[Egypt, Palestine, Syria]","[Autonomous Palestinian state on the West Bank and Gaza, Israel retains East Jerusalem, Israeli military outposts on...","(1, 1, 1)","(2, 1, 0)",57,78,"[x4, x6]","[i1, i4]","[Jordan, Saudi Arabia]","[Autonomous Palestinian state on the West Bank and Gaza, Israeli military outposts on the Golan Heights]","(0, 1, 1)","(1, 1, 0)",8,42
6,"[x2, x3, x5]","[i1, i3, i4]","[Egypt, Palestine, Syria]","[Autonomous Palestinian state on the West Bank and Gaza, Israel retains East Jerusalem, Israeli military outposts on...","(1, 1, 1)","(2, 1, 0)",57,78,"[x1, x4, x6]","[i1, i4]","[Israel, Jordan, Saudi Arabia]","[Autonomous Palestinian state on the West Bank and Gaza, Israeli military outposts on the Golan Heights]","(1, 1, 1)","(1, 1, 0)",57,42


,coalition_A_ids,coalition_B_ids,coalition_A_names,coalition_B_names,coalition_vU,coalition_vI,coalition_rU,coalition_rI,opposition_A_ids,opposition_B_ids,opposition_A_names,opposition_B_names,opposition_vU,opposition_vI,opposition_rU,opposition_rI
0,"[x2, x3, x5]","[i1, i3, i4]","[Egypt, Palestine, Syria]","[Autonomous Palestinian state on the West Bank and Gaza, Israel retains East Jerusalem, Israeli military outposts on...","(1, 1, 1)","(2, 1, 0)",57,78,[x1],"[i1, i2, i3, i4, i5]",[Israel],"[Autonomous Palestinian state on the West Bank and Gaza, Israeli military outposts along the Jordan River, Israel re...","(1, 0, 0)","(2, 2, 1)",49,85
1,"[x2, x3, x5]","[i1, i3, i4]","[Egypt, Palestine, Syria]","[Autonomous Palestinian state on the West Bank and Gaza, Israel retains East Jerusalem, Israeli military outposts on...","(1, 1, 1)","(2, 1, 0)",57,78,"[x1, x6]","[i1, i2, i4, i5]","[Israel, Saudi Arabia]","[Autonomous Palestinian state on the West Bank and Gaza, Israeli military outposts along the Jordan River, Israeli m...","(1, 1, 0)","(1, 2, 1)",56,49
2,"[x2, x3, x5]","[i1, i3, i4]","[Egypt, Palestine, Syria]","[Autonomous Palestinian state on the West Bank and Gaza, Israel retains East Jerusalem, Israeli military outposts on...","(1, 1, 1)","(2, 1, 0)",57,78,"[x1, x4, x6]","[i1, i4]","[Israel, Jordan, Saudi Arabia]","[Autonomous Palestinian state on the West Bank and Gaza, Israeli military outposts on the Golan Heights]","(1, 1, 1)","(1, 1, 0)",57,42


,coalition_A_ids,coalition_B_ids,coalition_A_names,coalition_B_names,coalition_vU,coalition_vI,coalition_rU,coalition_rI,opposition_A_ids,opposition_B_ids,opposition_A_names,opposition_B_names,opposition_vU,opposition_vI,opposition_rU,opposition_rI
0,"[x2, x3, x5]","[i1, i3, i4]","[Egypt, Palestine, Syria]","[Autonomous Palestinian state on the West Bank and Gaza, Israel retains East Jerusalem, Israeli military outposts on...","(1, 1, 1)","(2, 1, 0)",57,78,[x1],"[i1, i2, i3, i4, i5]",[Israel],"[Autonomous Palestinian state on the West Bank and Gaza, Israeli military outposts along the Jordan River, Israel re...","(1, 0, 0)","(2, 2, 1)",49,85


### Experiment 4 — compare semantic assumptions

The generalized code supports:
- agreement modes: `equal`, `equal_or_zero`
- opposition modes: `different`, `different_nonzero`


In [23]:
semantic_runs = []
for agreement_mode in ['equal', 'equal_or_zero']:
    for opposition_mode in ['different', 'different_nonzero']:
        m = pconf.ConflictModel.from_csv(
            attitudes_csv=DATA_DIR / 'attitudes.csv',
            agent_classes_csv=DATA_DIR / 'agent_classes.csv',
            issue_classes_csv=DATA_DIR / 'issue_classes.csv',
            agent_names_csv=DATA_DIR / 'agent_names.csv',
            issue_names_csv=DATA_DIR / 'issue_names.csv',
            agreement_mode=agreement_mode,
            opposition_mode=opposition_mode,
        )

        coalitions_i = m.enumerate_all_coalitions()
        q_i = pconf.frontier_Q(coalitions_i)
        qstar_i = pconf.frontier_Q_star(coalitions_i)
        oppositions_i = m.enumerate_oppositions_for_group(G)

        semantic_runs.append({
            'agreement_mode': agreement_mode,
            'opposition_mode': opposition_mode,
            'n_coalitions': len(coalitions_i),
            'n_frontier_q': len(q_i),
            'n_frontier_qstar': len(qstar_i),
            'n_oppositions_for_G': len(oppositions_i),
        })

df_semantics = pd.DataFrame(semantic_runs)
display(df_semantics.sort_values(['agreement_mode', 'opposition_mode']))


,agreement_mode,opposition_mode,n_coalitions,n_frontier_q,n_frontier_qstar,n_oppositions_for_G
0,equal,different,275,5,3,7
1,equal,different_nonzero,275,5,3,1
2,equal_or_zero,different,587,3,3,7
3,equal_or_zero,different_nonzero,587,3,3,1


### Experiment 5 — export results


In [24]:
OUT_DIR = PROJECT_DIR / 'MiddleEast' / 'Results'
OUT_DIR.mkdir(exist_ok=True)

pconf.export_structures_csv(OUT_DIR / 'coalitions.csv', coalitions, model)
pconf.export_structures_csv(OUT_DIR / 'coalitions_frontier_q.csv', front_q, model)
pconf.export_structures_csv(OUT_DIR / 'coalitions_frontier_qstar.csv', front_qstar, model)
pconf.export_structures_csv(OUT_DIR / 'oppositions.csv', oppositions, model)
pconf.export_structures_csv(OUT_DIR / 'oppositions_frontier_q.csv', opp_front_q, model)
pconf.export_structures_csv(OUT_DIR / 'oppositions_frontier_qstar.csv', opp_front_qstar, model)
pconf.export_biconflicts_csv(OUT_DIR / 'biconflicts.csv', biconflicts, model)
pconf.export_biconflicts_csv(OUT_DIR / 'biconflicts_frontier_q.csv', bi_front_q, model)
pconf.export_biconflicts_csv(OUT_DIR / 'biconflicts_frontier_qstar.csv', bi_front_qstar, model)

print('Exported to:', OUT_DIR.resolve())
sorted(p.name for p in OUT_DIR.iterdir())


Exported to: C:\Users\rafal.deja\Documents\Rafal\Praca naukowa\Project\ConflictAnalysis\MiddleEast\Results


['biconflicts.csv',
 'biconflicts_frontier_q.csv',
 'biconflicts_frontier_qstar.csv',
 'coalitions.csv',
 'coalitions_frontier_q.csv',
 'coalitions_frontier_qstar.csv',
 'oppositions.csv',
 'oppositions_frontier_q.csv',
 'oppositions_frontier_qstar.csv']

## Optional: run the CLI from the notebook

If you want to exercise the command-line interface from inside the notebook, uncomment and run the cell below.


In [ ]:
# import subprocess, sys
# cmd = [
#     sys.executable, 'pareto_conf_general_csv.py',
#     '--attitudes', 'MiddleEast/attitudes.csv',
#     '--agent-classes', 'MiddleEast/agent_classes.csv',
#     '--issue-classes', 'MiddleEast/issue_classes.csv',
#     '--agent-names', 'MiddleEast/agent_names.csv',
#     '--issue-names', 'MiddleEast/issue_names.csv',
#     '--coalitions', '--frontier', 'both',
#     '--group', 'x1', '--biconflicts',
# ]
# completed = subprocess.run(cmd, capture_output=True, text=True)
# print(completed.stdout)
# print(completed.stderr)
